Step 1: Identifying the Preciction Target

In [11]:
#%pip install numpy
#%pip install pandas
#%pip install -U scikit-learn
#%pip install matplotlib-inline


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip3.13 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip3.13 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip3.13 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
  Using cached matplotlib_inline-0.2.1-py3-none-any.whl.metadata (2.3 kB)
Using cached matplotlib_inline-0.2.1-py3-none-any.whl (9.5 kB)

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip3.13 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import numpy as np

In [7]:
df = pd.read_csv("bank-additional.csv", sep=";")
df.shape

(4119, 21)

In [22]:
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,30,blue-collar,married,basic.9y,no,yes,no,cellular,may,fri,...,2,999,0,nonexistent,-1.8,92.893,-46.2,1.313,5099.1,no
1,39,services,single,high.school,no,no,no,telephone,may,fri,...,4,999,0,nonexistent,1.1,93.994,-36.4,4.855,5191.0,no
2,25,services,married,high.school,no,yes,no,telephone,jun,wed,...,1,999,0,nonexistent,1.4,94.465,-41.8,4.962,5228.1,no
3,38,services,married,basic.9y,no,unknown,unknown,telephone,jun,fri,...,3,999,0,nonexistent,1.4,94.465,-41.8,4.959,5228.1,no
4,47,admin.,married,university.degree,no,yes,no,cellular,nov,mon,...,1,999,0,nonexistent,-0.1,93.200,-42.0,4.191,5195.8,no


The target variable is already labeled as "**y**" and it is a binary variable, leading me to think that this problem could be solved through logistic regression. Next I will separate it from the rest of the dataset. This variable is best for this prediction as it directly is the measure of weather the client subscribed or not. In the case of this marketing campaign, this is the main goal as we are trying to get as many customers to subscribe as possible.

In [14]:
X = df.drop("y", axis = 1)
y = df["y"]

From the dataset that 3 other variable could be seen as the target variable superficially. Firstly, '**duration**', which is the time in seconds that the call took. This variables should not be used as it does not really predict if the customer subscribed or not. This should be a predictor as it could even cause leakage as you only get this data after the phone call ended (after y). The second variable that could be considered a target is '**campaign**', which represents the number of contacts performed during this campaign for the client. Although this might seem like the target as it is directly related to the campaign, it does not indicate teh success of the campaign at all but rather if the campaign was operating efficiently. Finally, the last variable that seemed it could be the target to me is '**pdays**' which shows the number of days since last contact. This variable could seem as the target as one could infer that if the number of days is low, then the campaign did succeed. But again it isnt the target as it does not directly tell you if it succeeded or not, and again I think this could cause leakage.

Step 2: Data Loading and Exploration

In [2]:
import matplotlib.pyplot as plt

In [21]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4119 entries, 0 to 4118
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             4119 non-null   int64  
 1   job             4119 non-null   str    
 2   marital         4119 non-null   str    
 3   education       4119 non-null   str    
 4   default         4119 non-null   str    
 5   housing         4119 non-null   str    
 6   loan            4119 non-null   str    
 7   contact         4119 non-null   str    
 8   month           4119 non-null   str    
 9   day_of_week     4119 non-null   str    
 10  duration        4119 non-null   int64  
 11  campaign        4119 non-null   int64  
 12  pdays           4119 non-null   int64  
 13  previous        4119 non-null   int64  
 14  poutcome        4119 non-null   str    
 15  emp.var.rate    4119 non-null   float64
 16  cons.price.idx  4119 non-null   float64
 17  cons.conf.idx   4119 non-null   float64
 18 

In [20]:
df.dtypes

age                 int64
job                   str
marital               str
education             str
default               str
housing               str
loan                  str
contact               str
month                 str
day_of_week           str
duration            int64
campaign            int64
pdays               int64
previous            int64
poutcome              str
emp.var.rate      float64
cons.price.idx    float64
cons.conf.idx     float64
euribor3m         float64
nr.employed       float64
y                     str
dtype: object

We have 19 variables. 10 are numerical and 9 are categorical variables. The code above shows their specific type. From hindsight, the code says that there are no missing values but when looking at the table itself there are some, but they are labeled as unknown. I will now split the numerical and categorical variables

In [19]:
numerical_vars = X.select_dtypes(include = "number", exclude = "category")
categorical_vars = X.select_dtypes(include = "category", exclude = "number")

,age,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
0,30,487,2,999,0,-1.8,92.893,-46.2,1.313,5099.1
1,39,346,4,999,0,1.1,93.994,-36.4,4.855,5191.0
2,25,227,1,999,0,1.4,94.465,-41.8,4.962,5228.1
3,38,17,3,999,0,1.4,94.465,-41.8,4.959,5228.1
4,47,58,1,999,0,-0.1,93.200,-42.0,4.191,5195.8
